In [1]:

import os
import argparse
import json
from typing import Optional, Dict
from openai import OpenAI


In [3]:
os.environ["LLM_API_KEY"] = "sk-ajSWqeNQrsCIN4H1iuLcHEnjbJ1L90TKrPexyhgxbadR1QCg"           
os.environ["LLM_BASE_URL"] = "https://api.silra.cn/v1"  
os.environ["LLM_MODEL"] = "deepseek-chat"                  

In [7]:
LLM_API_KEY = os.getenv("LLM_API_KEY", "")
LLM_BASE_URL = os.getenv("LLM_BASE_URL", "")
LLM_MODEL = os.getenv("LLM_MODEL", "deepseek-chat")

TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))
TOP_P = float(os.getenv("TOP_P", "1.0"))
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "256"))

SYSTEM_PROMPT = "You are a helpful assistant. Follow the instruction and keep output concise."

def build_client() -> OpenAI:
    if not LLM_API_KEY:
        raise ValueError("缺少 LLM_API_KEY")
    # DeepSeek 属于 OpenAI 兼容服务：必须传 base_url
    return OpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL)

def llm(client: OpenAI, user_prompt: str) -> str:
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_TOKENS,
    )
    return (resp.choices[0].message.content or "").strip()

In [9]:
SENTENCE = {
    "p1": (
        "1. [Topic Reasoning]: Given S and I_CLIP, Based on common sense, what topic might it be discussing?  "
        "Return ONLY [R1].\n\nS: {S}\nI_CLIP: {I_CLIP}\n"
    ),
    "p2": (
        "2. [Opinion Reasoning]: Given S, I_CLIP and R1, what is the implicit opinion on the aforementioned topic and why? "
        "Return ONLY [R2].\n\nS: {S}\nI_CLIP: {I_CLIP}\nR1: {R1}\n"
    ),
    "p3": (
        "3. [Sentiment Reasoning]: Given S, I_CLIP and R2, Based on such an opinion, is the overall sentiment positive/neutral/negative? "
        "Return ONLY one word [R3].\n\nS: {S}\nI_CLIP: {I_CLIP}\nR2: {R2}\n"
    ),
}

ASPECT = {
    "p1": (
        "1. [Aspect Reasoning]: Given S, I_CLIP and A, which specific aspect of A is mentioned? "
        "Return ONLY [R1].\n\nS: {S}\nI_CLIP: {I_CLIP}\nA: {A}\n"
    ),
    "p2": (
        "2. [Opinion Reasoning]: Given S, I_CLIP, A and R1, Based on common sense, what is the implicit opinion towards the mentioned aspect of A, and why? "
        "Return ONLY [R2].\n\nS: {S}\nI_CLIP: {I_CLIP}\nA: {A}\nR1: {R1}\n"
    ),
    "p3": (
        "3. [Sentiment Reasoning]: Given S, I_CLIP, A and R2, Based on such opinion what is sentiment polarity towards A? "
        "Return ONLY one word [R3].\n\nS: {S}\nI_CLIP: {I_CLIP}\nA: {A}\nR2: {R2}\n"
    ),
}

def get_prompts(mode: str):
    return SENTENCE if mode == "sentence" else ASPECT

In [11]:

def run_hrf(
    client: OpenAI,
    mode: str,
    step: str,
    S: str,
    I_CLIP: str,
    A: Optional[str] = None,
    R1: Optional[str] = None,
    R2: Optional[str] = None,
) -> Dict[str, str]:
    prompts = get_prompts(mode)

    if mode == "aspect" and not A:
        raise ValueError("aspect 模式必须提供 A")

    out: Dict[str, str] = {}

    if step in ("step1", "all"):
        p1 = prompts["p1"].format(S=S, I_CLIP=I_CLIP, A=A or "")
        out["R1"] = llm(client, p1)

    if step in ("step2", "all"):
        r1 = out.get("R1") if step == "all" else (R1 or "")
        if not r1:
            raise ValueError("step2 需要 R1（你可以先跑 step1，或传入 --R1）")
        p2 = prompts["p2"].format(S=S, I_CLIP=I_CLIP, A=A or "", R1=r1)
        out["R2"] = llm(client, p2)

    if step in ("step3", "all"):
        r2 = out.get("R2") if step == "all" else (R2 or "")
        if not r2:
            raise ValueError("step3 需要 R2（你可以先跑 step2，或传入 --R2）")
        p3 = prompts["p3"].format(S=S, I_CLIP=I_CLIP, A=A or "", R2=r2)
        r3 = llm(client, p3).lower().strip()
        # 规整输出
        for lab in ("positive", "neutral", "negative"):
            if lab in r3:
                r3 = lab
                break
        out["R3"] = r3

    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["sentence", "aspect"], default="sentence")
    ap.add_argument("--step", choices=["step1", "step2", "step3", "all"], default="all")
    ap.add_argument("--S", required=True)
    ap.add_argument("--I_clip", required=True)
    ap.add_argument("--A", default=None)
    ap.add_argument("--R1", default=None)
    ap.add_argument("--R2", default=None)
    args = ap.parse_args()

    client = build_client()
    out = run_hrf(
        client=client,
        mode=args.mode,
        step=args.step,
        S=args.S,
        I_CLIP=args.I_clip,
        A=args.A,
        R1=args.R1,
        R2=args.R2,
    )
    print(json.dumps(out, ensure_ascii=False, indent=2))

#if __name__ == "__main__":
    #main()

In [15]:
client = build_client()

S = "I love this phone — the camera is amazing."
I_CLIP = "a person holding a smartphone and taking a photo"

out = run_hrf(client=client, mode="sentence", step="all", S=S, I_CLIP=I_CLIP)
print(out)
out1 = run_hrf(client=client, mode="sentence", step="step1", S=S, I_CLIP=I_CLIP)
R1 = out1["R1"]
out2 = run_hrf(client=client, mode="sentence", step="step2", S=S, I_CLIP=I_CLIP, R1=R1)
R2 = out2["R2"]
out3 = run_hrf(client=client, mode="sentence", step="step3", S=S, I_CLIP=I_CLIP, R2=R2)
print(out1, out2, out3)



{'R1': '[R1] Smartphone photography or camera quality.', 'R2': "[R2] The implicit opinion is that the smartphone has excellent camera capabilities, as the user's enthusiasm and specific praise for the camera quality directly relate to the depicted action of taking a photo.", 'R3': 'positive'}
{'R1': '[R1] Smartphone photography or camera quality.'} {'R2': "[R2] The implicit opinion is that the camera quality is excellent, as the user's enthusiasm and the act of taking a photo with the smartphone suggest a positive experience with its photographic capabilities."} {'R3': 'positive'}


In [17]:
S = "The battery lasts all day, but the camera is disappointing at night."
I_CLIP = "a smartphone on a table in a dimly lit room"
A = "battery, camera, screen"

out = run_hrf(client=client, mode="aspect", step="all", S=S, I_CLIP=I_CLIP, A=A)
print(out)
out1 = run_hrf(client=client, mode="sentence", step="step1", S=S, I_CLIP=I_CLIP)
R1 = out1["R1"]
out2 = run_hrf(client=client, mode="sentence", step="step2", S=S, I_CLIP=I_CLIP, R1=R1)
R2 = out2["R2"]
out3 = run_hrf(client=client, mode="sentence", step="step3", S=S, I_CLIP=I_CLIP, R2=R2)
print(out1, out2, out3)

{'R1': '[R1] camera', 'R2': '[R2] negative, because the camera is described as disappointing at night, which implies dissatisfaction with its performance in low-light conditions.', 'R3': 'negative'}
{'R1': '[R1] Smartphone review'} {'R2': '[R2] The implicit opinion is that the smartphone is adequate for daily use but falls short in low-light photography, as the battery life is praised while the camera performance is criticized in dim conditions.'} {'R3': 'negative'}
